In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


CUDA available: True
Device: Tesla T4


In [10]:
!pip install transformers datasets scikit-learn -q

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from torch.utils.data import Dataset
from sklearn.metrics import classification_report, f1_score

print("All imports successful")

All imports successful


In [4]:
train = pd.read_csv("https://raw.githubusercontent.com/HarshAggarwal524/hinglish-sentiment-analysis/main/data/emotion_train_final.csv")
dev = pd.read_csv("https://raw.githubusercontent.com/HarshAggarwal524/hinglish-sentiment-analysis/main/data/emotion_dev_final.csv")

print("Train:", train.shape)
print("Dev:", dev.shape)
print("\nTrain distribution:")
print(train['emotion'].value_counts())

Train: (5083, 3)
Dev: (897, 3)

Train distribution:
emotion
anger      2375
joy        1990
sadness     427
trust       291
Name: count, dtype: int64


In [5]:
# Label mapping
label2id = {'anger': 0, 'joy': 1, 'sadness': 2, 'trust': 3}
id2label = {0: 'anger', 1: 'joy', 2: 'sadness', 3: 'trust'}

train['label'] = train['emotion'].map(label2id)
dev['label'] = dev['emotion'].map(label2id)

print("Label distribution in train:")
print(train['label'].value_counts().sort_index())
print("\nAny nulls:", train['label'].isna().sum(), dev['label'].isna().sum())


Label distribution in train:
label
0    2375
1    1990
2     427
3     291
Name: count, dtype: int64

Any nulls: 0 0


In [11]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df['text'].tolist()
        self.labels = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = EmotionDataset(train, tokenizer)
dev_dataset = EmotionDataset(dev, tokenizer)

print(f"Train dataset: {len(train_dataset)}")
print(f"Dev dataset: {len(dev_dataset)}")

Train dataset: 5083
Dev dataset: 897


In [14]:
model = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=4,
    id2label=id2label,
    label2id=label2id
)

# Class weights to handle imbalance: anger=1.0, joy=1.0, sadness=2.5, trust=4.0
class_weights = torch.tensor([1.0, 1.0, 2.5, 4.0]).to('cuda')

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        from torch.nn import CrossEntropyLoss
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1_macro = f1_score(labels, predictions, average='macro')
    report = classification_report(
        labels, predictions,
        target_names=['anger', 'joy', 'sadness', 'trust']
    )
    print(report)
    return {'macro_f1': f1_macro}

training_args = TrainingArguments(
    output_dir="./xlmr_emotion",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=200,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    fp16=True
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics
)

print("Ready to train")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Ready to train


In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1
1,0.991462,0.911291,0.585793
2,0.762258,0.743302,0.627691
3,0.624953,0.751995,0.662875
4,0.430475,0.860712,0.673723
5,0.377897,0.900113,0.681331


              precision    recall  f1-score   support

       anger       0.82      0.90      0.86       419
         joy       0.82      0.83      0.82       351
     sadness       0.57      0.28      0.38        75
       trust       0.31      0.27      0.29        52

    accuracy                           0.78       897
   macro avg       0.63      0.57      0.59       897
weighted avg       0.77      0.78      0.77       897



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

       anger       0.86      0.86      0.86       419
         joy       0.89      0.74      0.81       351
     sadness       0.45      0.47      0.46        75
       trust       0.28      0.60      0.39        52

    accuracy                           0.76       897
   macro avg       0.62      0.67      0.63       897
weighted avg       0.80      0.76      0.78       897



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

              precision    recall  f1-score   support

       anger       0.90      0.82      0.85       419
         joy       0.88      0.85      0.86       351
     sadness       0.42      0.56      0.48        75
       trust       0.42      0.62      0.50        52

    accuracy                           0.80       897
   macro avg       0.65      0.71      0.67       897
weighted avg       0.82      0.80      0.81       897



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

       anger       0.89      0.81      0.85       419
         joy       0.86      0.86      0.86       351
     sadness       0.45      0.55      0.49        75
       trust       0.46      0.62      0.52        52

    accuracy                           0.80       897
   macro avg       0.66      0.71      0.68       897
weighted avg       0.82      0.80      0.80       897



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1590, training_loss=0.6996238672508384, metrics={'train_runtime': 821.4871, 'train_samples_per_second': 30.938, 'train_steps_per_second': 1.936, 'total_flos': 1671771887784960.0, 'train_loss': 0.6996238672508384, 'epoch': 5.0})

In [16]:
# Final evaluation
results = trainer.evaluate()
print(f"\nBest Macro F1: {results['eval_macro_f1']:.3f}")

              precision    recall  f1-score   support

       anger       0.89      0.81      0.85       419
         joy       0.86      0.86      0.86       351
     sadness       0.45      0.55      0.49        75
       trust       0.46      0.62      0.52        52

    accuracy                           0.80       897
   macro avg       0.66      0.71      0.68       897
weighted avg       0.82      0.80      0.80       897



Training Loss,Validation Loss,Epoch,Macro F1
0.377897,0.900113,5,0.681331



Best Macro F1: 0.681
